# OOD Static Metrics to Drive (Self-contained)

This notebook computes static metrics directly in Colab (no repo path/script dependency) and saves output CSV to Google Drive.


In [ ]:
%pip install -q nltk==3.8.1 pandas==2.2.0 textstat==0.7.3 spacy==3.7.4 numpy==1.26.4 tqdm
!python -m spacy download en_core_web_sm -q


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import nltk
import spacy
import textstat
from tqdm import tqdm
from nltk import pos_tag
from nltk.corpus import cmudict, stopwords, wordnet
from nltk.tokenize import sent_tokenize, word_tokenize

for resource in [
    'stopwords', 'cmudict', 'wordnet', 'averaged_perceptron_tagger',
    'punkt', 'punkt_tab', 'averaged_perceptron_tagger_eng'
]:
    nltk.download(resource, quiet=True)

nlp = spacy.load('en_core_web_sm')
STOP_WORDS = set(stopwords.words('english'))
CMU_DICT = cmudict.dict()

from google.colab import drive
drive.mount('/content/drive')

MY_DRIVE_SUBDIR = 'BeyondFK'  # change if needed
OOD_DIR = os.path.join('/content/drive/MyDrive', MY_DRIVE_SUBDIR, 'ood')
os.makedirs(OOD_DIR, exist_ok=True)

INPUT_CSV = os.path.join(OOD_DIR, 'onestopenglish_prepared.csv')
OUTPUT_CSV = os.path.join(OOD_DIR, 'onestopenglish_with_static.csv')

print('INPUT :', INPUT_CSV)
print('OUTPUT:', OUTPUT_CSV)


In [ ]:
def count_syllables(word):
    w = word.lower()
    if w in CMU_DICT:
        return len([ph for ph in CMU_DICT[w][0] if ph[-1].isdigit()])
    w = re.sub(r'[^a-z]', '', w)
    if not w:
        return 0
    count = 0
    vowels = 'aeiouy'
    prev = False
    for ch in w:
        is_v = ch in vowels
        if is_v and not prev:
            count += 1
        prev = is_v
    if w.endswith('e') and count > 1:
        count -= 1
    return max(count, 1)

def is_complex_word(word):
    return count_syllables(word) >= 3

def count_word_senses(word, pos=None):
    return len(wordnet.synsets(word, pos=pos))

def avg_distance_to_root(word, pos):
    synsets = wordnet.synsets(word, pos=pos)
    if not synsets:
        return 0
    return float(np.mean([s.min_depth() for s in synsets]))

AUXILIARY_VERBS = {
    'be','am','is','are','was','were','been','being','have','has','had','having',
    'do','does','did','will','would','shall','should','may','might','must','can','could'
}


In [ ]:
def compute_static_metrics(row):
    text = str(row.get('full_text', ''))
    question = str(row.get('text_question', ''))
    solution = str(row.get('text_solution', ''))
    lecture = str(row.get('text_lecture', ''))
    if not text.strip():
        return {m: 0 for m in STATIC_METRIC_NAMES}

    words = word_tokenize(text)
    words_alpha = [w for w in words if w.isalpha()]
    words_lower = [w.lower() for w in words_alpha]
    sentences = sent_tokenize(text)
    tagged = pos_tag(words_alpha)

    nouns = [w for w, t in tagged if t.startswith('NN')]
    verbs = [w for w, t in tagged if t.startswith('VB')]
    adjectives = [w for w, t in tagged if t.startswith('JJ')]
    adverbs = [w for w, t in tagged if t.startswith('RB')]
    doc = nlp(text)

    n_words_q = len(word_tokenize(question))
    n_words_a_solution = len(word_tokenize(solution))
    n_words_a_lecture = len(word_tokenize(lecture))
    text_length = len(text)
    word_count = len(words_alpha)

    num_numbers = sum(1 for c in text if c.isdigit())
    num_commas = text.count(',')
    num_complex = sum(1 for w in words_alpha if is_complex_word(w))
    num_unique = len(set(words_lower))

    content_pos_words = set(w.lower() for w in nouns + verbs + adjectives + adverbs)
    num_content = len([w for w in words_lower if w in content_pos_words])
    content_words_no_stop = [w for w in words_lower if w not in STOP_WORDS]
    num_content_no_stop = len(content_words_no_stop)

    syllables = [count_syllables(w) for w in words_alpha]
    avg_syllables = float(np.mean(syllables)) if syllables else 0.0
    sent_lengths = [len(word_tokenize(s)) for s in sentences]
    avg_sent_len = float(np.mean(sent_lengths)) if sent_lengths else 0.0

    num_prep_phrases = sum(1 for token in doc if token.pos_ == 'ADP')
    num_negated_stem = sum(1 for token in doc if token.dep_ == 'neg')
    neg_words = {'no','not','never','neither','nobody','nothing','nowhere','nor'}
    num_negated_lead_in = sum(1 for w in words_lower if w in neg_words)

    noun_chunks = list(doc.noun_chunks)
    num_main_np = len(noun_chunks)
    np_lengths = [len(chunk) for chunk in noun_chunks]
    avg_np_length = float(np.mean(np_lengths)) if np_lengths else 0.0
    num_verb_phrases = len([token for token in doc if token.pos_ == 'VERB'])

    passive_verbs, active_verbs = set(), set()
    for token in doc:
        if token.dep_ == 'nsubjpass':
            passive_verbs.add(token.head.i)
        if token.dep_ == 'nsubj' and token.head.pos_ == 'VERB':
            active_verbs.add(token.head.i)
    passive_count = len(passive_verbs)
    active_count = len(active_verbs)

    agentless_passive = 0
    for token in doc:
        if token.dep_ == 'nsubjpass':
            has_agent = any(child.dep_ == 'agent' for child in token.head.children)
            if not has_agent:
                agentless_passive += 1

    total_voice = active_count + passive_count
    prop_active = active_count / total_voice if total_voice > 0 else 1.0
    prop_passive = passive_count / total_voice if total_voice > 0 else 0.0
    ratio_active_passive = (active_count / passive_count) if passive_count > 0 else float(active_count)

    words_before_verb_list = []
    for sent in doc.sents:
        for i, token in enumerate(sent):
            if token.pos_ == 'VERB' and token.dep_ in ('ROOT', 'ccomp', 'advcl'):
                words_before_verb_list.append(i)
                break
    words_before_verb = float(np.mean(words_before_verb_list)) if words_before_verb_list else 0.0

    word_len_std = float(np.std([len(w) for w in words_alpha])) if words_alpha else 0.0
    unique_words_lower = set(words_lower)
    num_polysemic = sum(1 for w in unique_words_lower if count_word_senses(w) > 1)
    total_senses = sum(count_word_senses(w) for w in unique_words_lower)

    unique_content_no_stop = set(content_words_no_stop)
    content_word_senses = sum(count_word_senses(w) for w in unique_content_no_stop)
    noun_senses = sum(count_word_senses(w.lower(), wordnet.NOUN) for w in set(nouns))
    verb_senses = sum(count_word_senses(w.lower(), wordnet.VERB) for w in set(verbs))
    non_aux_verbs = [w for w in verbs if w.lower() not in AUXILIARY_VERBS]
    non_aux_verb_senses = sum(count_word_senses(w.lower(), wordnet.VERB) for w in set(non_aux_verbs))
    adj_senses = sum(count_word_senses(w.lower(), wordnet.ADJ) for w in set(adjectives))
    adv_senses = sum(count_word_senses(w.lower(), wordnet.ADV) for w in set(adverbs))
    noun_depths = [avg_distance_to_root(w.lower(), wordnet.NOUN) for w in set(nouns)]
    verb_depths = [avg_distance_to_root(w.lower(), wordnet.VERB) for w in set(verbs)]
    dist_root_nouns = float(np.mean(noun_depths)) if noun_depths else 0.0
    dist_root_verbs = float(np.mean(verb_depths)) if verb_depths else 0.0

    fk_grade = textstat.flesch_kincaid_grade(text)
    fk_ease = textstat.flesch_reading_ease(text)
    cl_index = textstat.coleman_liau_index(text)
    ari = textstat.automated_readability_index(text)
    smog = textstat.smog_index(text)
    gunning = textstat.gunning_fog(text)
    prop_prep = num_prep_phrases / word_count if word_count > 0 else 0.0
    traenkle_bailer = 224.6814 - (79.8304 * (avg_sent_len / 100.0)) - (12.24032 * (prop_prep * 100.0)) if word_count > 0 else 0.0

    return {
        'n_words_q': n_words_q, 'n_words_a_solution': n_words_a_solution, 'n_words_a_lecture': n_words_a_lecture,
        'Text_Length': text_length, 'Word_Count': word_count,
        'Nouns': len(nouns), 'Verbs': len(verbs), 'Adjectives': len(adjectives), 'Adverbs': len(adverbs),
        'Num_Numbers': num_numbers, 'Num_Commas': num_commas,
        'Num_Complex_Words': num_complex, 'Num_Unique_Words': num_unique,
        'Num_Content_Words': num_content, 'Num_Content_Words_No_Stopwords': num_content_no_stop,
        'Word_Length_Syllables': avg_syllables, 'Avg_Sentence_Length': avg_sent_len,
        'Num_Prepositional_Phrases': num_prep_phrases, 'Num_Negated_Words_Stem': num_negated_stem,
        'Num_Negated_Words_Lead_In': num_negated_lead_in, 'Num_Main_Noun_Phrases': num_main_np,
        'Avg_Main_NP_Length': avg_np_length, 'Num_Verb_Phrases': num_verb_phrases,
        'Prop_Active_Voice_Verbs': prop_active, 'Prop_Passive_Voice_Verbs': prop_passive,
        'Ratio_Active_to_Passive_Verbs': ratio_active_passive, 'Num_Words_Before_Main_Verb': words_before_verb,
        'Num_Agentless_Passive_Constructions': agentless_passive, 'Word_Length_Std_Dev': word_len_std,
        'Num_Polysemic_Words': num_polysemic, 'Num_Word_Senses': total_senses,
        'Num_Word_Senses_For_Content_Words': content_word_senses, 'Num_Word_Senses_For_Nouns': noun_senses,
        'Num_Word_Senses_For_Verbs': verb_senses, 'Num_Word_Senses_For_Non_Auxiliary_Verbs': non_aux_verb_senses,
        'Num_Word_Senses_For_Adjectives': adj_senses, 'Num_Word_Senses_For_Adverbs': adv_senses,
        'Distance_To_Root_Nouns': dist_root_nouns, 'Distance_To_Root_Verbs': dist_root_verbs,
        'flesch_kincaid_grade': fk_grade, 'flesch_kincaid_ease': fk_ease,
        'coleman_liau_index': cl_index, 'automated_readability_index': ari,
        'smog_index': smog, 'gunning_fog': gunning, 'traenkle_bailer_index': traenkle_bailer,
    }

STATIC_METRIC_NAMES = list(compute_static_metrics({'full_text': 'bootstrap', 'text_question': '', 'text_solution': '', 'text_lecture': ''}).keys())
print('Metric count:', len(STATIC_METRIC_NAMES))


In [ ]:
assert os.path.exists(INPUT_CSV), f'Input CSV not found: {INPUT_CSV}'
df = pd.read_csv(INPUT_CSV)
if 'full_text' not in df.columns:
    if 'text' not in df.columns:
        raise ValueError('Need full_text or text column in input CSV')
    df['full_text'] = df['text'].astype(str)
for c in ('text_question', 'text_solution', 'text_lecture'):
    if c not in df.columns:
        df[c] = ''
    else:
        df[c] = df[c].fillna('').astype(str)

rows = []
for i in tqdm(range(len(df)), desc='Computing static metrics'):
    rows.append(compute_static_metrics(df.iloc[i]))

metrics_df = pd.DataFrame(rows)
out = pd.concat([df.reset_index(drop=True), metrics_df], axis=1)
out.to_csv(OUTPUT_CSV, index=False)
print('Saved:', OUTPUT_CSV)
print('Shape:', out.shape)
out.head(2)
